In [78]:
import pandas as pd

In [79]:
customers = pd.read_csv("customers_cleaned.csv")

In [80]:
customers["acquisition_date"] = pd.to_datetime(customers["acquisition_date"])
customers["churn_date"] = pd.to_datetime(customers["churn_date"])

In [81]:
cohort_data = customers[["customer_id", "acquisition_date", "churn_date"]]
cohort_data.head()

,customer_id,acquisition_date,churn_date
0,C0000001,2026-01-15,2026-03-22
1,C0000002,2014-01-01,NaT
2,C0000003,2015-11-11,2015-12-22
3,C0000004,2014-10-18,NaT
4,C0000005,2014-10-20,2023-10-22


In [83]:
cohort_data["cohort_date"] = cohort_data["acquisition_date"].dt.to_period("M")
cohort_data = cohort_data[["customer_id", "acquisition_date", "cohort_date", "churn_date"]].copy()
cohort_data.head()

,customer_id,acquisition_date,cohort_date,churn_date
0,C0000001,2026-01-15,2026-01,2026-03-22
1,C0000002,2014-01-01,2014-01,NaT
2,C0000003,2015-11-11,2015-11,2015-12-22
3,C0000004,2014-10-18,2014-10,NaT
4,C0000005,2014-10-20,2014-10,2023-10-22


In [84]:
today = pd.Timestamp.today()
cohort_data["effective_churn_date"] = cohort_data["churn_date"].fillna(today)
cohort_data = cohort_data[["customer_id", "acquisition_date", "cohort_date", "effective_churn_date"]]
cohort_data.head()

,customer_id,acquisition_date,cohort_date,effective_churn_date
0,C0000001,2026-01-15,2026-01,2026-03-22 00:00:00.000000
1,C0000002,2014-01-01,2014-01,2026-08-06 14:26:51.136633
2,C0000003,2015-11-11,2015-11,2015-12-22 00:00:00.000000
3,C0000004,2014-10-18,2014-10,2026-08-06 14:26:51.136633
4,C0000005,2014-10-20,2014-10,2023-10-22 00:00:00.000000


In [85]:
cohort_data["lifetime_months"] = ((cohort_data["effective_churn_date"] - cohort_data["acquisition_date"]).dt.days / 30).astype(int)
cohort_data[["customer_id", "acquisition_date", "cohort_date", "effective_churn_date", "lifetime_months"]]
cohort_data.head()

,customer_id,acquisition_date,cohort_date,effective_churn_date,lifetime_months
0,C0000001,2026-01-15,2026-01,2026-03-22 00:00:00.000000,2
1,C0000002,2014-01-01,2014-01,2026-08-06 14:26:51.136633,153
2,C0000003,2015-11-11,2015-11,2015-12-22 00:00:00.000000,1
3,C0000004,2014-10-18,2014-10,2026-08-06 14:26:51.136633,143
4,C0000005,2014-10-20,2014-10,2023-10-22 00:00:00.000000,109


In [86]:
cohort_data["Active_3M"] = (cohort_data["lifetime_months"] >= 3).astype(int)

In [87]:
cohort_data["Active_6M"] = (cohort_data["lifetime_months"] >= 6).astype(int)

In [88]:
cohort_data["Active_12M"] = (cohort_data["lifetime_months"] >= 12).astype(int)

In [89]:
cohort_data[["customer_id","lifetime_months","Active_3M","Active_6M","Active_12M"]].head()

,customer_id,lifetime_months,Active_3M,Active_6M,Active_12M
0,C0000001,2,0,0,0
1,C0000002,153,1,1,1
2,C0000003,1,0,0,0
3,C0000004,143,1,1,1
4,C0000005,109,1,1,1


In [96]:
cohort_retention = (cohort_data.pivot_table(index="cohort_date",values=["Active_3M","Active_6M","Active_12M"],aggfunc="mean")*100).round(2)
cohort_retention

,Active_12M,Active_3M,Active_6M
cohort_date,,,
2014-01,99.33,99.85,99.66
2014-02,100.00,100.00,100.00
2014-03,100.00,100.00,100.00
2014-04,100.00,100.00,100.00
2014-05,98.95,98.95,98.95
...,...,...,...
2025-11,0.00,77.88,69.91
2025-12,0.00,74.53,69.81
2026-01,0.00,56.64,56.64


## Observation

- Customer retention decreases from the 3-month period to the 12-month period across all acquisition cohorts, indicating normal customer churn over time.
- Earlier acquisition cohorts generally show better long-term retention compared to newer cohorts.
- Cohort retention analysis helps identify customer groups with lower retention, enabling targeted retention strategies to improve customer loyalty.